# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset name:', metadata.name)
print('\nDescription:')
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their field `@id`s
record_sets = list(dataset.record_sets)

if not record_sets:
    print('No record sets found in dataset metadata.')
else:
    print('Found the following record sets:')
    for rset in record_sets:
        print(f"- Record set: {rset['@id']}")
        fields = rset.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print('  Field @id(s):')
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif isinstance(f, str):
                print(f"    - {f}")
else:
    # If no record sets are listed, guide the user
    print('The metadata does not provide loaded record sets. You may need to examine records by direct IDs if known.')

# Optionally, if there are no record sets, print available distributions as a hint
if not record_sets and hasattr(metadata, 'distribution'):
    print('\nAvailable distributions:')
    try:
        for dist in metadata.distribution:
            print(f"- distribution @id: {getattr(dist, '@id', dist)}")
    except Exception as e:
        print(metadata.distribution)

## 3. Data Extraction
Load data from available record sets into a DataFrame for analysis.

If no record sets are configured in the Croissant schema, we can try loading the data from available distributions directly using their `@id`.

In [ ]:
# Select record set(s) by @id, or load from distributions if record sets are missing
record_set_ids = [rset['@id'] for rset in dataset.record_sets] if list(dataset.record_sets) else []

dataframes = {}

if record_set_ids:
    print('Extracting data from record sets:')
    for rsid in record_set_ids:
        try:
            records = list(dataset.records(record_set=rsid))
            dataframes[rsid] = pd.DataFrame(records)
            print(f"Record set '{rsid}' loaded: {len(dataframes[rsid])} rows, {len(dataframes[rsid].columns)} columns.")
        except Exception as e:
            print(f"Could not load record set '{rsid}': {str(e)}")
else:
    # Fallback: Attempt to access available distributions
    if hasattr(metadata, 'distribution') and metadata.distribution:
        print('No record sets found in metadata. Attempting to extract records from distributions.')
        distributions = metadata.distribution
        try:
            for dist in distributions:
                dist_id = getattr(dist, '@id', str(dist))
                try:
                    records = list(dataset.records(distribution=dist_id))
                    if records:
                        dataframes[dist_id] = pd.DataFrame(records)
                        print(f"Loaded from distribution {dist_id}: {len(records)} records.")
                except Exception as e:
                    print(f"Could not extract records from distribution {dist_id}: {e}")
        except Exception as e:
            print('Error when iterating over distributions:', e)

# Display columns for each loaded DataFrame
for key, df in dataframes.items():
    print(f"\nFirst columns for '{key}':")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, handling missing values, normalizing numeric fields, and grouping by key attributes.

If no record sets are defined, choose a distribution DataFrame for analysis. Replace below with the actual IDs and columns identified above.

In [ ]:
# Identify a DataFrame to analyze
if dataframes:
    # Pick the first DataFrame loaded
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    print(f"Analyzing data from: {df_key}")

    # List possible numeric fields for analysis
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    print('Numeric columns:', numeric_cols)

    # Select a numeric field (replace with known column @id if possible)
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")

        # Filter outliers using a threshold (example: keep values above median)
        try:
            threshold = df[numeric_field].median()
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalize the chosen field
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
                filtered_df[numeric_field].std())
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by a categorical column if present
            possible_groups = df.select_dtypes(include='object').columns.tolist()
            group_field = possible_groups[0] if possible_groups else None
            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"Grouped (mean) {numeric_field} by {group_field}:")
                display(grouped_df.head())
        except Exception as e:
            print('Error in EDA step:', e)
    else:
        print('No numeric fields found to analyze.')
else:
    print('No dataframes available for EDA analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        field = numeric_cols[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {field}')
        plt.xlabel(field)
        plt.ylabel('Count')
        plt.show()

        # Plot relationship with a categorical variable if present
        cat_cols = df.select_dtypes(include='object').columns.tolist()
        if cat_cols:
            group_field = cat_cols[0]
            plt.figure(figsize=(10,4))
            sns.boxplot(data=df, x=group_field, y=field)
            plt.title(f'{field} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(field)
            plt.xticks(rotation=45)
            plt.show()
    else:
        print('No numeric fields available for plotting.')
else:
    print('No dataframes found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded dataset metadata and attempted to extract available records.
- Key numeric fields were identified and used for example filtering, normalization, and group-wise statistics.
- Simple visualizations illustrated data distributions and field relationships.
- The FAIR^2 dataset provides insight into knowledge adoption and rangeland management across northern Kenya. For advanced analysis, further curation of record sets and deeper data cleaning may be beneficial.